# Diagnóstico de cancro da mama — Exploração dos dados

**Inteligência Artificial Centrada no Humano** · Mestrado em Inteligência Artificial e Engenharia de Dados · FCTUC

Conhecer o dataset antes de treinar: quantas imagens há por classe, como são, e quanta memória ocupam consoante o tamanho escolhido (ponto levantado pelo professor: as imagens podem ser pesadas e pode ser preciso limitar o dataset).

> **Estado**
> - ✅ Feito: código completo e testado.
> - ⬜ Falta: correr e escrever as conclusões (tamanho de imagem a usar).

**Índice**
1. Configuração
2. Distribuição das classes
3. Exemplos de imagens
4. Tamanho das imagens
5. Balanceamento
6. Memória necessária
7. Conclusões

## 1. Configuração

A primeira célula prepara o ambiente e funciona tanto no **Google Colab** como **localmente**:
- no Colab, monta o Google Drive (onde deve estar uma cópia do repositório com o dataset) e instala as bibliotecas que faltam (`modAL-python`, `scikeras`);
- localmente, assume que o notebook foi aberto a partir da pasta `notebooks/`.

Depois junta a raiz do repositório ao `sys.path`, para se poder importar o código partilhado da pasta `src/`. Os caminhos do dataset e da pasta de resultados estão em `src/config.py`.

In [ ]:
# Ambiente: monta o Google Drive no Colab; localmente usa a pasta do repositório
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/miacd-iach-breast-cancer-diagnosis")  # pasta do repositório no Drive
    %pip install -q modAL-python scikeras
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
from src.config import DATA_PATH, RESULTS_PATH

## 2. Distribuição das classes

O dataset é o [Breast Ultrasound Images](https://datasetninja.com/breast-ultrasound-images): ecografias mamárias de mulheres entre os 25 e os 75 anos, divididas em três classes:
- **normal**: sem lesão;
- **benign**: lesão benigna (não cancerígena);
- **malignant**: lesão maligna (cancro).

A classe de cada imagem vem do nome do ficheiro (ex.: `benign (12).png`).

O dataset tem um erro: `benign (433).png` e `malignant (145).png` são a mesma imagem com classes diferentes. Como não se sabe qual está certa, as duas são excluídas quando se carregam os dados.

In [ ]:
from src.exploration import class_counts, list_images, plot_class_distribution

names, labels = list_images(DATA_PATH)
counts = class_counts(labels)
print(f"Total: {len(names)} imagens", counts)
plot_class_distribution(counts, "Dataset original")

from src.data import find_duplicates
print("Imagens duplicadas (excluídas no carregamento):", sorted(find_duplicates(DATA_PATH)))

O dataset é desequilibrado: há mais do triplo de imagens benignas do que normais. Sem balanceamento, o modelo tende a favorecer a classe maioritária: pode acertar muito só por dizer "benigno" quase sempre, e falhar justamente os casos malignos, que são os que mais importam.

## 3. Exemplos de imagens

Algumas imagens aleatórias de cada classe. Repara que as diferenças entre classes não são óbvias a olho nu: as lesões aparecem como zonas escuras, e a forma e os contornos (regulares ou irregulares) são o que distingue uma lesão benigna de uma maligna. Algumas imagens têm também anotações do ecógrafo (texto, marcas), que o modelo pode aprender por engano — é uma das razões para usar explicabilidade (Grad-CAM) mais tarde.

In [ ]:
from src.exploration import plot_examples

plot_examples(DATA_PATH, names, labels, per_class=4)

## 4. Tamanho das imagens

As imagens originais têm tamanhos diferentes, mas uma CNN precisa que todas as entradas tenham o mesmo tamanho. Por isso são todas redimensionadas para `IMG_SIZE × IMG_SIZE` (quadradas) no carregamento.

Quanto maior o `IMG_SIZE`, mais detalhe se mantém, mas mais memória e tempo de treino são precisos.

In [ ]:
import numpy as np
from src.exploration import original_sizes

sizes = original_sizes(DATA_PATH, names)
print(f"Largura: {sizes[:, 0].min()}–{sizes[:, 0].max()} px (mediana {int(np.median(sizes[:, 0]))})")
print(f"Altura:  {sizes[:, 1].min()}–{sizes[:, 1].max()} px (mediana {int(np.median(sizes[:, 1]))})")

## 5. Balanceamento

A divisão treino/validação/teste (65/15/20, estratificada) é feita **antes** de balancear, para que versões rodadas da mesma imagem não fiquem em treino e em validação/teste (data leakage). Só o treino é balanceado, com rotações e reflexões (`src/data.py`):

| Classe | Imagens originais | Cópias acrescentadas | Fator |
|---|---|---|---|
| normal | 133 | 5 (2 rotações, 1 reflexão, 2 rotações com reflexão) | ×6 |
| malignant | 210 | 3 (1 rotação, 1 reflexão, 1 rotação com reflexão) | ×4 |
| benign | 437 | 1 (reflexão) | ×2 |

Rodar ou espelhar uma ecografia não muda o diagnóstico, por isso estas cópias são exemplos válidos e dão ao modelo mais variedade. Os gráficos abaixo mostram as contagens antes e depois.

In [ ]:
from src.data import load_dataset

IMG_SIZE = 64   # pequeno, só para ver as contagens rapidamente
x_train, x_val, x_test, y_train, y_val, y_test = load_dataset(DATA_PATH, size=IMG_SIZE)
plot_class_distribution(class_counts(list(y_train)), "Treino depois do balanceamento")
plot_class_distribution(class_counts(list(y_val)), "Validação (imagens originais)")
plot_class_distribution(class_counts(list(y_test)), "Teste (imagens originais)")

## 6. Memória necessária

Memória ocupada só pelas imagens (cada píxel é um `float32` = 4 bytes, 1 canal porque é escala de cinzento), sem contar o modelo nem o treino, que usam bastante mais. Ajuda a escolher `IMG_SIZE` nos notebooks seguintes: o professor alertou que as imagens podem ser pesadas. O Colab gratuito tem cerca de 12 GB de RAM.

In [ ]:
from src.exploration import memory_mb

n_images = len(x_train) + len(x_val) + len(x_test)
for size in [64, 128, 224]:
    print(f"{size}×{size}: {memory_mb(n_images, size):6.0f} MB")

## 7. Conclusões

*A preencher: tamanho de imagem escolhido e se é preciso limitar o dataset.*